# Inference Demo

**Objective:** Demonstrate end-to-end video and image inference using trained models.

**Research Traceability:**
- Research Objective: Frame-level and video-level deepfake classification
- Methodology: Real-time prediction with confidence scoring and aggregation
- Implementation: notebooks/07_inference_demo.ipynb

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.inference.predict_image import ImagePredictor
from src.inference.predict_video import VideoPredictor
from src.inference.frame_analysis import FrameAnalyzer

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Image Prediction

In [ ]:
# Initialize image predictor
model_path = Path('../outputs/models/best_xceptionnet.pth')

if model_path.exists():
    predictor = ImagePredictor(
        model_path=str(model_path),
        model_type='xceptionnet',
        device='cpu'
    )
    print("Image predictor initialized")
else:
    print("Model not found. Train the model first.")
    predictor = None

In [ ]:
def demo_image_prediction(image_path: str):
    """Demo image prediction with visualization."""
    if predictor is None:
        print("Predictor not available")
        return
    
    # Load and display image
    import cv2
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Get prediction
    result = predictor.predict(image_path)
    
    # Display
    plt.figure(figsize=(8, 8))
    plt.imshow(image_rgb)
    plt.title(f'Prediction: {result["label"]}\n'
              f'Confidence: {result["confidence"]:.2%}')
    plt.axis('off')
    plt.show()
    
    return result

# Test on sample image
sample_image = '../data/sample/test_real.jpg'
if Path(sample_image).exists():
    result = demo_image_prediction(sample_image)
    print(f"\nPrediction result: {result}")

## 2. Video Prediction

In [ ]:
def demo_video_prediction(video_path: str):
    """Demo video prediction with frame-by-frame analysis."""
    if predictor is None:
        print("Predictor not available")
        return
    
    video_predictor = VideoPredictor(
        model_path=str(model_path),
        model_type='xceptionnet',
        device='cpu',
        aggregation='average'
    )
    
    # Get prediction
    result = video_predictor.predict(video_path)
    
    print(f"Video: {video_path}")
    print(f"Label: {result['label']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Frames analyzed: {result.get('num_frames', 'N/A')}")
    
    return result

## 3. Frame Analysis Timeline

In [ ]:
def demo_frame_analysis(video_path: str):
    """Demo frame-level analysis with timeline visualization."""
    analyzer = FrameAnalyzer(
        model_path=str(model_path),
        model_type='xceptionnet',
        device='cpu'
    )
    
    # Analyze video
    result = analyzer.analyze(video_path)
    
    # Plot timeline
    if 'frame_scores' in result:
        scores = result['frame_scores']
        
        plt.figure(figsize=(12, 4))
        plt.plot(scores, marker='o', markersize=3)
        plt.axhline(y=0.5, color='r', linestyle='--', label='Decision Boundary')
        plt.xlabel('Frame Index')
        plt.ylabel('Fake Probability')
        plt.title('Frame-Level Deepfake Detection Timeline')
        plt.legend()
        plt.tight_layout()
        plt.savefig('../outputs/plots/frame_analysis_timeline.png', dpi=150)
        plt.show()
    
    return result

## 4. Explainability with GradCAM

In [ ]:
from src.visualization.explainability import GradCAM

def demo_gradcam(image_path: str):
    """Demo GradCAM visualization."""
    gradcam = GradCAM(
        model_path=str(model_path),
        model_type='xceptionnet',
        device='cpu'
    )
    
    # Generate heatmap
    heatmap = gradcam.generate(image_path)
    
    # Display
    import cv2
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image_rgb)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(heatmap, cmap='jet')
    plt.title('GradCAM Heatmap')
    plt.axis('off')
    
    plt.tight_layout()
    plt.savefig('../outputs/plots/gradcam_example.png', dpi=150)
    plt.show()

## Summary
- Image prediction with confidence scoring
- Video prediction with frame aggregation
- Frame-level analysis timeline
- GradCAM explainability visualization

## Next Steps
- Deploy with FastAPI: `src/api/app.py`
- Generate thesis assets: `scripts/generate_thesis_assets.py`